In [1]:
import pandas as pd
import numpy as np
import duckdb
import sys
from pathlib import Path

In [2]:
# Paths 

main = Path("/kellogg/proj/lgg3230")
rais_aux = main / "UnionSpill/Data/RAIS_aux"

# DAta paths:


clauses_path = rais_aux / "cba_clauses_by_period.dta"
bilateral_path = rais_aux / "bilateral_connectivity_2007_2011.csv"
output_path = rais_aux / "cba_similarity_panel.dta"

In [3]:
clauses = pd.read_stata(str(clauses_path), convert_categoricals=False)
clauses.head()

,identificad,cl_0,cl_0not_iden,cl_11des_sal,cl_11iso_sal,cl_11pis_sal,cl_11rea_cor,cl_12pag_sal,cl_13rem_dsr,cl_13sal_est,...,cl_91apl_ins,cl_91des_ins,cl_91mec_sol,cl_91out_dis,cl_91reg_par,cl_91ren_res,lagos_sample_avg,treat_ultra,in_balanced_panel,cba_period
0,00001961000110,0,0,1,0,1,1,1,0,0,...,2,1,0,3,0,0,1,1,1,1.0
1,00001961000110,0,0,1,0,1,1,1,0,0,...,2,1,0,3,0,0,1,1,1,2.0
2,00001961000110,0,0,1,0,1,1,1,0,0,...,2,1,0,3,0,0,1,1,1,3.0
3,00001961000110,0,0,1,0,1,1,1,0,0,...,2,1,0,3,0,0,1,1,1,4.0
4,00001961000110,0,0,1,0,1,1,1,0,0,...,2,1,0,3,0,0,1,1,1,5.0


In [4]:
clause_vars = [c for c in clauses.columns if c.startswith("cl_")]
print(f"  {len(clause_vars)} clause variables, {len(clauses)} observations")


  139 clause variables, 89735 observations


In [5]:
clauses.isna().sum()

identificad          0
cl_0                 0
cl_0not_iden         0
cl_11des_sal         0
cl_11iso_sal         0
                    ..
cl_91ren_res         0
lagos_sample_avg     0
treat_ultra          0
in_balanced_panel    0
cba_period           0
Length: 144, dtype: int64

In [6]:
# Collapse to firm × cba_period
clauses = clauses.groupby(["identificad", "cba_period"]).agg(
    {**{v: "mean" for v in clause_vars},
     "treat_ultra":       "max",
     "lagos_sample_avg":  "max",
     "in_balanced_panel": "max"}
).reset_index()

treated_ids   = set(clauses.loc[clauses.treat_ultra == 1, "identificad"])
untreated_ids = set(clauses.loc[clauses.treat_ultra == 0, "identificad"])
print(f"  Treated firms: {len(treated_ids):,}  |  Untreated firms: {len(untreated_ids):,}")

  Treated firms: 13,202  |  Untreated firms: 4,634


In [9]:
# ── Load bilateral connectivity ───────────────────────────────────────────────
print("Loading bilateral connectivity...", flush=True)
bilateral = pd.read_csv(str(bilateral_path))
bilateral["id_i"] = bilateral["identificad_i"].astype("int64").astype(str).str.zfill(15).str[1:]
bilateral["id_j"] = bilateral["identificad_j"].astype("int64").astype(str).str.zfill(15).str[1:]

bilateral["weight"] = pd.to_numeric(bilateral["bilateral_conn_pw"], errors="coerce").fillna(0)
bilateral = bilateral[bilateral["weight"] > 0][["id_i", "id_j", "weight"]].copy()
print(f"  {len(bilateral):,} bilateral pairs with positive weight")
bilateral['weight'].describe()

Loading bilateral connectivity...
  38,634 bilateral pairs with positive weight


count    38634.000000
mean         0.038127
std          0.066595
min          0.000071
25%          0.003454
50%          0.017544
75%          0.048925
max          2.000000
Name: weight, dtype: float64

In [10]:
# ── (untreated firm i) → (treated firm j) directed pairs ──────────────────────
fwd = bilateral[
    bilateral["id_i"].isin(untreated_ids) & bilateral["id_j"].isin(treated_ids)
].rename(columns={"id_i": "firm_i", "id_j": "firm_j"})

rev = bilateral[
    bilateral["id_j"].isin(untreated_ids) & bilateral["id_i"].isin(treated_ids)
].rename(columns={"id_j": "firm_i", "id_i": "firm_j"})

bilateral_ut = pd.concat([fwd, rev], ignore_index=True).drop_duplicates(
    subset=["firm_i", "firm_j"]
)
print(f"  Untreated→treated pairs: {len(bilateral_ut):,}")
print(f"  Untreated firms with at least one treated connection: "
      f"{bilateral_ut['firm_i'].nunique():,}", flush=True)

  Untreated→treated pairs: 6,581
  Untreated firms with at least one treated connection: 1,721
